# SI_Tweet_Text_cleaned.ipynb

**Purpose:** Produce Tables S16/S17 (sentiment by gender) and Figs. S23-S26
(word clouds, tag distributions).

**Outputs** (figure in `figures/` <-> source data in `tables/`, same name):

| Figure | Source data |
|---|---|
| `fig_s23_female.pdf`, `fig_s23_male.pdf` | `fig_s23_female.csv`, `fig_s23_male.csv` |
| `fig_s24.pdf` | `fig_s24.csv` (histogram bars), `fig_s24_summary.csv` |
| `fig_s25_female.pdf`, `fig_s25_male.pdf` | `fig_s25_female.csv`, `fig_s25_male.csv` |
| `fig_s26.pdf` | `fig_s26.csv` (histogram bars), `fig_s26_summary.csv` |
| -- | `table_s16.csv`, `table_s17.csv` |

## Changes from your latest version

**Fixed:**
- `text` was being read via `pd.read_csv()`, which cannot preserve a Python
  list. Tested directly: this silently breaks `.explode()` later --
  `text_clean` ends up as the literal string `"['tweet a', 'tweet b']"`
  instead of two separate tweets, and no error is raised. Fixed with
  `ast.literal_eval()` right after the read to restore real lists.
- `author_12_16_sample_tw` was never defined (the old broken
  self-referencing line was removed, but nothing replaced it) -- added as
  `author_text.copy()`.
- `py_path` (undefined) and `nlp_test`/`classifer` (undefined) -- already
  fixed in your latest version; kept as-is.

**Also removed** (unchanged from the last cleanup): exploratory-only cells
(language-detection trials, a 10-row test run), the old duplicate
`sentiment_pip()`, and the dead `.mode(axis=1)` block that would error if run.

## Required input

`clean_for_Git/TW_text.csv` (your anonymized file), with columns:
`author_id`, `doi_lower`, `gender`, `self_pro`, `self_text`, `text`, `title_en`
-- where `text` is a **string-encoded list** (e.g. `"['tweet A', 'tweet B']"`),
parsed back into a real list on load.

## Required external dependency (not included)

`Py/nlp_test.py`, `Py/sentiment_analysis.py` + their pre-trained model files.


In [ ]:
import ast
import re
import collections

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

data_dir = "1_data"
results_dir = "2_result"   # intermediate files
figures_dir = "figures"    # every manuscript figure
tables_dir = "tables"      # figure source data + supplementary tables

import os
os.makedirs(figures_dir, exist_ok=True)
os.makedirs(tables_dir, exist_ok=True)

pd.set_option("max_colwidth", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

### 1. Load sentiment models

In [ ]:
import pickle
import Py.sentiment_analysis as sa
import Py.nlp_test as nlp_test

NB_model = pickle.load(open(f"{data_dir}/Py/sentiment_NB_model.sav", "rb"))
LR_model = pickle.load(open(f"{data_dir}/Py/sentiment_LR_model.sav", "rb"))
tf_vector = pickle.load(open(f"{data_dir}/Py/sentiment_tf_vector.pk", "rb"))

classifer = nlp_test.model_traning()

In [ ]:
import torch
from transformers import pipeline

sentiment_pipeline = pipeline("sentiment-analysis", model="finiteautomata/bertweet-base-sentiment-analysis")

### 2. Load the anonymized dataset

In [ ]:
author_12_16_tw = pd.read_csv(f"{data_dir}/clean_for_Git/TW_text.csv")

# NOTE on a fix: `text` (a list of tweets per publication) does not survive a
# CSV round-trip as a real Python list -- pd.read_csv() loads it as the
# literal string "[\'tweet A\', \'tweet B\']". Restore it here, since
# .explode() later requires a genuine list.
author_12_16_tw["text"] = author_12_16_tw["text"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

required_cols = ["author_id", "doi_lower", "gender", "self_pro", "self_text", "text", "title_en"]
missing = [c for c in required_cols if c not in author_12_16_tw.columns]
if missing:
    raise ValueError(f"author_12_16_tw is missing required columns: {missing}")

print(f"Loaded author_12_16_tw: {len(author_12_16_tw)} rows")
print("text column is now a list?", isinstance(author_12_16_tw["text"].dropna().iloc[0], list))

### 3. Text-cleaning helper functions

In [ ]:
def clean_text(text, clean):
    if clean == 1:
        x = " ".join(re.sub(r"(@[A-Za-z0-9]+)|([^0-9A-Za-z \t])|(\w+://\S+)", " ", text).split())
        x = x.lower()
    else:
        x = text.lower()
    return x


def clean_tweet(tweet, clean):
    if isinstance(tweet, list):
        return [clean_text(i, clean) for i in tweet if isinstance(i, str)]
    elif isinstance(tweet, str):
        return clean_text(tweet, clean)
    return "0"


def remove_html(text):
    return re.sub(r"http\S+", "", text)


def remove_at(text):
    return re.sub(r"@\S+", "", text)


def analyze_tag(text):
    return text.count("@"), text.count("#")


def process_tweet(text):
    tag_1, tag_2 = analyze_tag(text)
    text_pro = remove_at(remove_html(text))
    return tag_1, tag_2, text_pro

### 4. Build the two analysis samples

> **Fix applied:** `author_12_16_sample_tw` was used later (e.g. `.explode()`)
> but never defined anywhere -- added explicitly as `author_text.copy()`.

In [ ]:
author_text = author_12_16_tw[author_12_16_tw["text"].notnull()].copy()
author_text["text_clean"] = author_text["text"].apply(lambda x: clean_tweet(x, 0))
author_text["self_text_clean"] = author_text["self_text"].apply(lambda x: clean_tweet(x, 0))

author_12_16_sample_tw = author_text.copy()

author_12_16_sample_tw_test = author_12_16_sample_tw.explode("text_clean", ignore_index=True)
author_12_16_sample_tw_test = author_12_16_sample_tw_test[author_12_16_sample_tw_test["text_clean"].notnull()].copy()
author_12_16_sample_tw_test["at_tag"], author_12_16_sample_tw_test["hash_tag"], author_12_16_sample_tw_test["self_text_proce"] = zip(
    *author_12_16_sample_tw_test["text_clean"].apply(process_tweet)
)

author_12_16_sample_self = author_12_16_sample_tw[
    (author_12_16_sample_tw["self_pro"] == 1) & (author_12_16_sample_tw["self_text"].notnull())
].copy()
author_12_16_sample_self["at_tag"], author_12_16_sample_self["hash_tag"], author_12_16_sample_self["self_text_proce"] = zip(
    *author_12_16_sample_self["self_text"].apply(process_tweet)
)

print(f"author_12_16_sample_tw_test: {len(author_12_16_sample_tw_test)} rows")
print(f"author_12_16_sample_self: {len(author_12_16_sample_self)} rows")

### 5. Sentiment scoring

In [ ]:
import langid
from translate import Translator

pd.set_option("display.max_colwidth", None)


def sentiment_pip(sentence, title, LR_model, tf_vector):
    sentence_list = sentence.lower().split()

    if isinstance(title, str):
        title_list = title.lower().split()
        for word in title_list:
            if word in sentence_list:
                sentence_list.remove(word)
        sentence_new = " ".join(sentence_list)
    else:
        sentence_new = sentence.lower()

    if sentence_new.isspace() or sentence_new == "":
        return "None", sentence_new, "Neutral", 1, "Neutral", "Neutral"

    lang = langid.classify(sentence_new)[0]
    sentence_trans = sentence_new
    if lang != "en":
        try:
            sentence_trans = Translator(from_lang=lang, to_lang="en").translate(sentence_new)
        except Exception:
            sentence_trans = sentence_new

    bert_sentiment = sentiment_pipeline(sentence_trans, truncation=True, max_length=128)[0]
    senti_sa = sa.text_analysis(sentence_trans, LR_model, tf_vector)
    senti_nlp = nlp_test.test_sentiment(sentence_trans, classifer)

    return lang, sentence_new, bert_sentiment["label"], bert_sentiment["score"], senti_nlp, senti_sa

In [ ]:
author_12_16_sample_tw_test["language"], author_12_16_sample_tw_test["senti_sentence"], \
    author_12_16_sample_tw_test["senti"], author_12_16_sample_tw_test["senti_score"], \
    author_12_16_sample_tw_test["senti_nlp"], author_12_16_sample_tw_test["senti_sa"] = zip(
    *author_12_16_sample_tw_test.apply(lambda row: sentiment_pip(row["self_text_proce"], row["title_en"], LR_model, tf_vector), axis=1)
)

sent_cols = ["senti", "senti_nlp", "senti_sa"]
author_12_16_sample_tw_test[sent_cols] = author_12_16_sample_tw_test[sent_cols].apply(
    lambda col: col.str.strip().str.capitalize()
)

In [ ]:
author_12_16_sample_self["language"], author_12_16_sample_self["senti_sentence"], \
    author_12_16_sample_self["senti"], author_12_16_sample_self["senti_score"], \
    author_12_16_sample_self["senti_nlp"], author_12_16_sample_self["senti_sa"] = zip(
    *author_12_16_sample_self.apply(lambda row: sentiment_pip(row["self_text_proce"], row["title_en"], LR_model, tf_vector), axis=1)
)

author_12_16_sample_self[sent_cols] = author_12_16_sample_self[sent_cols].apply(
    lambda col: col.str.strip().str.capitalize()
)

### 6. Tables S16 and S17: sentiment by gender

In [ ]:
from scipy.stats import chi2_contingency


def classify_sentiment(row, cols=("senti", "senti_nlp", "senti_sa"),
                        score_col="senti_score", high_conf_threshold=0.9):
    values = row[list(cols)]
    counts = values.value_counts()
    if counts.iloc[0] > 1:
        return counts.idxmax()
    if row[score_col] > high_conf_threshold:
        return row["senti"]
    return "Neutral"


SENTIMENT_LABEL_MAP = {
    "Neg": "Negative", "Negative": "Negative", "NEG": "Negative",
    "Neu": "Neutral", "Neutural": "Neutral", "Neutral": "Neutral", "NEU": "Neutral",
    "Pos": "Positive", "Positive": "Positive", "POS": "Positive",
}


def clean_sentiment_labels(df, col="senti_majority"):
    df = df.copy()
    df[col] = df[col].str.strip().map(SENTIMENT_LABEL_MAP)
    unmapped = df[col].isna().sum()
    if unmapped > 0:
        raise ValueError(f"{unmapped} rows had an unmapped sentiment label.")
    return df


def wilson_ci(x, n, z=1.959964):
    p = x / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half_width = (z / denom) * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return center - half_width, center + half_width


def newcombe_ci(x1, n1, x2, n2, z=1.959964):
    p1, p2 = x1 / n1, x2 / n2
    l1, u1 = wilson_ci(x1, n1, z)
    l2, u2 = wilson_ci(x2, n2, z)
    diff = p1 - p2
    lower = diff - np.sqrt((p1 - l1) ** 2 + (u2 - p2) ** 2)
    upper = diff + np.sqrt((u1 - p1) ** 2 + (p2 - l2) ** 2)
    return lower, upper


def cohens_h(p1, p2):
    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))


def format_p(p):
    return "<.001" if p < 0.001 else f"{p:.3f}"


def sentiment_gender_table(df, sentiment_col="senti_majority", gender_col="gender",
                            female_label="female", male_label="male",
                            sentiment_order=("Negative", "Neutral", "Positive")):
    df = clean_sentiment_labels(df, sentiment_col)
    counts = df.groupby([gender_col, sentiment_col]).size().unstack(gender_col).fillna(0)
    n_female, n_male = counts[female_label].sum(), counts[male_label].sum()

    rows = []
    for sentiment in sentiment_order:
        x_f, x_m = counts.loc[sentiment, female_label], counts.loc[sentiment, male_label]
        p_f, p_m = x_f / n_female, x_m / n_male
        table = [[x_f, n_female - x_f], [x_m, n_male - x_m]]
        _, p_val, _, _ = chi2_contingency(table)
        ci_low, ci_high = newcombe_ci(x_f, n_female, x_m, n_male)
        h = cohens_h(p_f, p_m)
        rows.append({
            "Sentiment": sentiment,
            "Female, n (%)": f"{int(x_f):,} ({100*p_f:.2f}%)",
            "Male, n (%)": f"{int(x_m):,} ({100*p_m:.2f}%)",
            "Diff (F-M), pp": round(100 * (p_f - p_m), 2),
            "95% CI": f"[{100*ci_low:.2f}, {100*ci_high:.2f}]",
            "Cohen's h": round(h, 3),
            "P-value": format_p(p_val),
        })
    return pd.DataFrame(rows)

In [ ]:
author_12_16_sample_tw_test["senti_majority"] = author_12_16_sample_tw_test.apply(classify_sentiment, axis=1)
tie_rate = author_12_16_sample_tw_test.apply(
    lambda r: r[["senti", "senti_nlp", "senti_sa"]].nunique() == 3, axis=1
).mean()
print(f"Online mentions: 3-way disagreement rate = {tie_rate:.2%}")

table_S16 = sentiment_gender_table(author_12_16_sample_tw_test)
table_S16.to_csv(f"{tables_dir}/table_s16.csv", index=False)
table_S16

In [ ]:
author_12_16_sample_self["senti_majority"] = author_12_16_sample_self.apply(classify_sentiment, axis=1)
tie_rate = author_12_16_sample_self.apply(
    lambda r: r[["senti", "senti_nlp", "senti_sa"]].nunique() == 3, axis=1
).mean()
print(f"Self-promoted: 3-way disagreement rate = {tie_rate:.2%}")

table_S17 = sentiment_gender_table(author_12_16_sample_self)
table_S17.to_csv(f"{tables_dir}/table_s17.csv", index=False)
table_S17

### 7. Figs. S24 and S26: tag distributions

Fig. S24 = tags in all tweet mentions; Fig. S26 = tags in self-promoted tweets.
Each figure's source data (`tables/fig_s24.csv`, `tables/fig_s26.csv`) holds the
exact histogram bars drawn (bin edges, count, proportion per gender x tag type);
`*_summary.csv` holds the descriptive statistics (N, mean, SD, median, IQR, min, max).

In [ ]:
def tag_summary_table(df, tags=("at_tag", "hash_tag"), gender_col="gender"):
    rows = []
    for gender, g_df in df.groupby(gender_col):
        for tag in tags:
            data = g_df[tag]
            rows.append({
                "Gender": gender,
                "Tag type": tag,
                "N": len(data),
                "Mean": round(data.mean(), 3),
                "SD": round(data.std(), 3),
                "Median": data.median(),
                "IQR": f"[{data.quantile(0.25):.0f}, {data.quantile(0.75):.0f}]",
                "Min": data.min(),
                "Max": data.max(),
            })
    return pd.DataFrame(rows)


tag_summary_table(author_12_16_sample_tw_test).to_csv(f"{tables_dir}/fig_s24_summary.csv", index=False)
tag_summary_table(author_12_16_sample_self).to_csv(f"{tables_dir}/fig_s26_summary.csv", index=False)

In [ ]:
def plot_tag_distribution(df, fig_path, csv_path):
    """Draw the 2x2 tag-distribution histograms (female/male x @/#) and write
    the exact bars drawn (taken from ax.hist's return values) as source data."""
    rows = []
    m = 0
    fig = plt.figure(figsize=(10, 8))
    for gender in df["gender"].unique():
        colors, G = ("#6C9BCF", "Female") if gender == "female" else ("#F79540", "Male")
        for tag in ["at_tag", "hash_tag"]:
            n_bins = 50 if tag == "at_tag" else 10
            m += 1
            ax = plt.subplot(2, 2, m)
            data = df[df["gender"] == gender][tag].to_list()
            heights, edges, _ = ax.hist(data, weights=np.ones(len(data)) / len(data), bins=n_bins,
                                        color=colors, edgecolor="black", align="left")
            plt.ylim([0, 1])
            ax.set_title(f"{G} : distribution of {tag} related tweets", size=8)
            ax.set_xlabel("The number of tags", size=10)
            ax.set_ylabel("Proportion", size=10)
            for k, h in enumerate(heights):
                rows.append({
                    "Gender": G, "Tag type": tag,
                    "Bin left": edges[k], "Bin right": edges[k + 1],
                    "Count": int(round(h * len(data))), "Proportion": h, "N": len(data),
                })
    plt.tight_layout()
    plt.savefig(fig_path, bbox_inches="tight")
    plt.show()
    pd.DataFrame(rows).to_csv(csv_path, index=False)


# Fig. S24: all tweet mentions
plot_tag_distribution(author_12_16_sample_tw_test, f"{figures_dir}/fig_s24.pdf", f"{tables_dir}/fig_s24.csv")

# Fig. S26: self-promoted tweets
plot_tag_distribution(author_12_16_sample_self, f"{figures_dir}/fig_s26.pdf", f"{tables_dir}/fig_s26.csv")

### 8. Figs. S23 and S25: word clouds

In [ ]:
from wordcloud import WordCloud
from textblob import TextBlob

STOPWORDS = ["is", "are", "be", "to", "for", "in", "the", "of", "on", "your", "a", "-",
             "as", "and", "at", "from", "new", "have", "has", "that", "it", "this",
             "article", "paper", "publish", "published", "out", "journal", "publication",
             "now", "was", "were", "by", "with", "i", "been"]
STOPWORDS = [w.lower() for w in STOPWORDS]

wrd = WordCloud(width=480, height=480, margin=0, collocations=False)
for sw in STOPWORDS:
    wrd.stopwords.add(sw)


def get_adjectives(text):
    blob = TextBlob(text)
    return " ".join(word for word, tag in blob.tags if tag == "JJ")


def clean_word_list(values):
    cleaned = (re.sub(r"[^A-Za-z0-9]", "", str(x)) for x in values)
    return [s for s in cleaned if len(s) > 1]


def make_wordcloud_figure(text_values, subtitle, csv_path, fig_path):
    words_clean = clean_word_list(text_values)
    text = " ".join(words_clean)
    wordcloud = wrd.generate(text)

    filtered_words = [w for w in text.split() if w not in STOPWORDS]
    counted_words = collections.Counter(filtered_words)
    total_count = sum(counted_words.values())

    words, counts, proportions = [], [], []
    for word, count in counted_words.most_common(15):
        words.append(word)
        counts.append(count)
        proportions.append(count / total_count if total_count else 0)

    colors = cm.rainbow(np.linspace(0, 1, len(words)))

    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 6), gridspec_kw={"width_ratios": [1.2, 1]})
    axes[0].imshow(wordcloud, interpolation="bilinear")
    axes[0].axis("off")
    axes[0].set_title("Word cloud of tweet mentions", fontsize=14)

    axes[1].barh(words, proportions, color=colors)
    axes[1].set_xlabel("Proportion", fontsize=12)
    axes[1].set_ylabel("Words", fontsize=12)
    axes[1].set_title(subtitle, fontsize=14)
    plt.tight_layout()
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    word_table = pd.DataFrame({"Word": words, "Count": counts, "Proportion": [round(p, 4) for p in proportions]})
    word_table.insert(0, "Rank", range(1, len(word_table) + 1))
    word_table.to_csv(csv_path, index=False)


author_12_16_sample_self["senti_sentence_adj"] = author_12_16_sample_self["senti_sentence"].apply(get_adjectives)
author_12_16_sample_tw_test["senti_sentence_adj"] = author_12_16_sample_tw_test["senti_sentence"].apply(get_adjectives)

In [ ]:
make_wordcloud_figure(
    author_12_16_sample_self[author_12_16_sample_self["gender"] == "female"]["senti_sentence_adj"].values,
    "Top words in female researchers\u2019\nself-promoted Tweets",
    f"{tables_dir}/fig_s25_female.csv", f"{figures_dir}/fig_s25_female.pdf",
)
make_wordcloud_figure(
    author_12_16_sample_self[author_12_16_sample_self["gender"] == "male"]["senti_sentence_adj"].values,
    "Top words in male researchers\u2019\nself-promoted Tweets",
    f"{tables_dir}/fig_s25_male.csv", f"{figures_dir}/fig_s25_male.pdf",
)
make_wordcloud_figure(
    author_12_16_sample_tw_test[author_12_16_sample_tw_test["gender"] == "female"]["senti_sentence_adj"].values,
    "Top words in tweet mentions on female researchers\u2019\nfirst publications",
    f"{tables_dir}/fig_s23_female.csv", f"{figures_dir}/fig_s23_female.pdf",
)
make_wordcloud_figure(
    author_12_16_sample_tw_test[author_12_16_sample_tw_test["gender"] == "male"]["senti_sentence_adj"].values,
    "Top words in tweet mentions on male researchers\u2019\nfirst publications",
    f"{tables_dir}/fig_s23_male.csv", f"{figures_dir}/fig_s23_male.pdf",
)